# 04 · Energía de un sitio nuevo

Con un lugar, un año y una potencia, la aplicación estima la energía del año. El clima sale del parque más cercano: el medido si ese año existe, o el promedio mensual anterior a 2024. La energía es factor estimado × potencia (MW) × horas del mes.

Este notebook usa la misma función que valida el dashboard, sobre la grilla precalculada de 0,2°.

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    import subprocess
    root = Path('/content/tpo-solar')
    if not root.exists():
        subprocess.run(['git','clone','https://github.com/imelinc/tpo-solar.git',str(root)],check=True)
    os.chdir(root)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, DATA_PROCESSED
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# La evidencia la escribe python -m src.academico. Con REENTRENAR = True se recalcula acá (tarda unos minutos).
REENTRENAR = False
ruta = DATA_PROCESSED / 'defensa_evidencia.json'
if REENTRENAR or not ruta.exists():
    from src.academico import construir
    E = construir()
else:
    E = json.loads(ruta.read_text())
R = E['regresion']
print('Elegido por validación:', R['nombre'])

In [ ]:
from src.academico import consultar
args = (E['grilla'], E['parques'], E['clima'], E['tipico'], E['rango'])
filas = []
for p in E['presets']:
    r = consultar(*args, p['lat'], p['lon'], p['anio'], p['potencia_mw'])
    filas.append({'caso':p['nombre'], 'GWh':round(r['energia_gwh'],2), 'factor':round(r['factor'],3), 'clima':r['modo'],
                  'dentro del rango':r['dentro'], 'parque que presta el clima':r['nombre'], 'km':round(r['distancia_km'])})
pd.DataFrame(filas)

## Cómo se arma el total

Cada mes multiplica el factor estimado por la potencia y por las horas de ese mes. Duplicar la potencia duplica la energía.

In [ ]:
r = consultar(*args, E['presets'][0]['lat'], E['presets'][0]['lon'], 2023, 10)
meses = pd.DataFrame(r['meses'])
meses['horas'] = meses['gwh'] * 1000 / (meses['factor'] * 10)
print(meses.round(3))
print('Total:', round(meses['gwh'].sum(), 2), 'GWh')
px.bar(meses, x='mes', y='gwh', color='clima', title='Energía estimada por mes, 10 MW, 2023').show()

## El caso límite

Ushuaia queda fuera del rectángulo de los parques de entrenamiento. La consulta cae en la celda del borde y el modelo no extrapola: el número no vale como estimación, aunque sea mayor que el de Catamarca.

In [ ]:
u = consultar(*args, -54.8, -68.3, 2023, 10)
print({k: u[k] for k in ['energia_gwh','dentro','lat_celda','lon_celda','distancia_celda_km','nombre']})